## 1. 高级用法1：设置Agent的名称

In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

agent = create_agent(model = model_openai, name = "chat_assistant")

result = agent.invoke({
    "messages": ["你好"]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================
Name: chat_assistant

你好！很高兴为你服务。请问有什么我可以帮你的吗？


## 2.高级用法2：系统提示词
使用 create_agent 创建 Agent 时，需传入`模型` 和`工具`、可选地传入`系统提示词`。提示词为Agent提供了任务背景、行为准则和操作指南。

系统指令，即SystemMessage，通过`system_prompt` 设置，定义Agent行为。这个参数可以是`str`或者`SystemMessage类型` 。

使用建议：
- 明确说明 Agent 的角色
- 定义输出格式
- 说明何时使用工具

举例1：

In [8]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain.messages import SystemMessage
from langchain.tools import tool
from rich import print as rprint
import os

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool(name_or_callable="add_number_tool")
def add_number(a:int, b:int) -> str:
    """
    计算并返回两个数的和。
    Args:
        a: 加数
        b: 加数

    Returns:
        总和
    """
    return f"总和为：{a+b}"

#创建客服助手Agent
agent = create_agent(
    model = model_openai,
    name = "add_number_agent",
    tools = [add_number],
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "10加上20，再加上30是多少？"}
    ]
})

rprint(result)

# print(result["messages"][-1].content)


{
    'messages': [
        HumanMessage(
            content='10加上20，再加上30是多少？',
            additional_kwargs={},
            response_metadata={},
            id='e82ea695-06b5-436a-80ee-5fdfe0e39d23'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 446,
                    'prompt_tokens': 327,
                    'total_tokens': 773,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 405,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 446
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 327
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-67da8713-5d05-953c-b837-f5f4d9fcbbee',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='add_number_agent',
            id='lc_run--01a0be8d-22f3-78b2-b79b-fea4160c47e5-0',
            tool_calls=[
                {
                    'name': 'add_number_tool',
                    'args': {'a': 10, 'b': 20},
                    'id': 'call_4e62c62f1d22421582f5ab87',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 327,
                'output_tokens': 446,
                'total_tokens': 773,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 405}
            }
        ),
        ToolMessage(
            content='总和为：30',
            name='add_number_tool',
            id='139708f3-5c13-4e5f-9ae0-8b3c17fad5b7',
            tool_call_id='call_4e62c62f1d22421582f5ab87'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 94,
                    'prompt_tokens': 388,
                    'total_tokens': 482,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 53,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 94
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 388
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-a975aa82-7d6e-97f4-a8ca-5180dd6b7bc2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='add_number_agent',
            id='lc_run--01a0be8d-3cdd-74b0-b1c6-370f1bcf60d6-0',
            tool_calls=[
                {
                    'name': 'add_number_tool',
                    'args': {'a': 30, 'b': 30},
                    'id': 'call_e1606fe04e7a4c13a9e26f4d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 388,
                'output_tokens'

举例2：

In [9]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv(override=True)

model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

flag = 0

@tool(name_or_callable="get_weather_tool")
def get_weather(city: str) -> str:
    """
    天气工具查询

    Args:
        city: 城市名称

    Returns:
        今天天气很好，气温28°C
    """
    global flag
    flag += 1

    if flag < 3:
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"
    return f"{city}今天天气很好，气温28°C"

messages = [
    HumanMessage("你好，深圳今天的天气如何？")
]

agent = create_agent(
    model=model_openai,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应该再次调用同一个工具，最多重试3次。"
        "如果3次后仍然失败，再向用户说明服务暂时不可用。"
    )
)

response = agent.invoke({"messages": messages})

for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

你好，深圳今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather_tool (call_d16b45b0345347f0a04ba879)
 Call ID: call_d16b45b0345347f0a04ba879
  Args:
    city: 深圳
================================= Tool Message =================================
Name: get_weather_tool

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather_tool (call_304a4162057e473d99cd0489)
 Call ID: call_304a4162057e473d99cd0489
  Args:
    city: 深圳
================================= Tool Message =================================
Name: get_weather_tool

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather_tool (call_1368b7a0e9a64f349aad39cf)
 Call ID: call_1368b7a0e9a64f349aad39cf
  Args:
    city: 深圳
===========

## 3. 高级用法3：结构化输出
结构化输出是Agent的核心功能之一，它允许Agent以特定、可预测的格式返回数据，而不是传统的自然语言响应。通过结构化输出，开发者可以直接获得`Pydantic模型` 、 `JSON对象` 或 `数据类` 等结构化数据，这些数据能够被应用程序直接使用，无需复杂的解析过程。

结构化输出的4种策略：

① ProviderStrategy

In [10]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1. 模型初始化
model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# 2. Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """
    用户的联系方式
    """
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户电子邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3. agent初始化
agent = create_agent(
    model = model_openai,
    response_format=ProviderStrategy(ContactInfo),
)

# 4.嗲用
response = agent.invoke({"messages": [
    HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：boyeofmountain@qq.com，手机哈是13312345678")
]})

for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：boyeofmountain@qq.com，手机哈是13312345678
================================== Ai Message ==================================

{
  "name": "小明",
  "email": "boyeofmountain@qq.com",
  "phone": "13312345678"
}


② ToolStrategy

In [15]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1. 模型初始化
model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# 2. Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """
    用户的联系方式
    """
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户电子邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3. 工具的定义
@tool
def search_tool(query: str) -> str:
    """
    这是一个搜索引擎。当大模型发现给定的上下文里缺少必要的联系人信息，需要去互联网上查询时，才会调用这个工具。
    """
    return f"搜索结果: 未找到关于 '{query}' 的更多额外信息。"

agent = create_agent(
    model = model_openai,
    tools = [search_tool],
    response_format = ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "联系人信息：Bobby What，boye@123.com，(0755)87217821"}
    ]
})

#print(result["structured_response"])
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

联系人信息：Bobby What，boye@123.com，(0755)87217821
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_e6c1ef5fb8f54318a8737d36)
 Call ID: call_e6c1ef5fb8f54318a8737d36
  Args:
    name: Bobby What
    email: boye@123.com
    phone: (0755)87217821
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='Bobby What' email='boye@123.com' phone='(0755)87217821'


③ type / AutoStrategy

In [18]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.structured_output import AutoStrategy
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1. 模型初始化
model_openai = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

# 2. Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """
    用户的联系方式
    """
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户电子邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3. 工具的定义
@tool
def search_tool(query: str) -> str:
    """
    这是一个搜索引擎。当大模型发现给定的上下文里缺少必要的联系人信息，需要去互联网上查询时，才会调用这个工具。
    """
    return f"搜索结果: 未找到关于 '{query}' 的更多额外信息。"

agent = create_agent(
    model = model_openai,
    tools = [search_tool],
    response_format = AutoStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "联系人信息：Bobby What，boye@123.com，(0755)87217821"}
    ]
})

#print(result["structured_response"])
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

联系人信息：Bobby What，boye@123.com，(0755)87217821
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_d562137fa8e347768439ad79)
 Call ID: call_d562137fa8e347768439ad79
  Args:
    name: Bobby What
    email: boye@123.com
    phone: (0755)87217821
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='Bobby What' email='boye@123.com' phone='(0755)87217821'


④ None: 默认配置，表示不以结构化输出，以 自然语言 响应用户问题。

总结：

在实际大模型Agent开发场景中，如果使用到了结构化输出，推荐使用 “ToolStrategy”策略 ，所以后续重点介绍这种策略方式结构化输出。